# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will inspect the record sets (tables) defined in the Croissant schema and then show their available fields and column `@id`s.


In [ ]:
# Display available record sets and their fields
from pprint import pprint

record_sets = dataset.metadata.record_sets

print("Record Sets in the dataset:")
for rs in record_sets:
    print(f"- Record Set @id: {rs['@id']} | name: {rs.get('name', 'No name')} | description: {rs.get('description', '')}")
    fields = rs.get('field', [])
    print("  Fields/Columns:")
    for field in fields:
        # Try getting field name and id
        field_name = field.get('name', field.get('@id'))
        print(f"    * @id: {field['@id']}, name: {field_name}, dataType: {field.get('dataType', '')}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will demonstrate how to extract all tables and show example contents.

In [ ]:
# Extract data from each record set by @id
dfs = {}

# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dfs[record_set_id] = df

# Show columns and head from the first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"Columns for record set {first_rs}: {dfs[first_rs].columns.tolist()}")
    display(dfs[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

First, let's select the appropriate record set and some numeric and categorical fields for demonstration.


In [ ]:
# Example EDA on the first available record set
from IPython.display import display

# Pick the first record set
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dfs[record_set_id]

    # Find a numeric field
    rs = next((rs for rs in dataset.metadata.record_sets if rs['@id'] == record_set_id), None)
    numeric_field_id = None
    group_field_id = None
    for field in rs.get('field', []):
        dtype = str(field.get('dataType', '')).lower()
        if 'int' in dtype or 'float' in dtype or dtype=='number':
            numeric_field_id = field['@id']
            break
    # Find a group-able (categorical) field
    for field in rs.get('field', []):
        dtype = str(field.get('dataType', '')).lower()
        if 'text' in dtype or 'string' in dtype:
            group_field_id = field['@id']
            break

    # If numeric field found, filter and normalize
    if numeric_field_id and numeric_field_id in df.columns:
        threshold = None
        try:
            # Try to guess a threshold
            threshold = df[numeric_field_id].mean()
        except Exception:
            threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by categorical field, if present
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA in the first record set.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll create a histogram of the normalized numeric field and a boxplot grouped by the categorical field if those are present in the record set.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only visualize if numeric field found
if record_set_ids and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in Record Set {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(9,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the FAIR^2 dataset using its Croissant schema, reviewed its structure via record sets and fields (using their `@id`). We extracted data for each record set, explored numeric and categorical fields, filtered and normalized values, and visualized distributions.

Due to the small sample size and specific cohort (N=3 female patients), generalizations are limited. Fields and relationships are clearly labeled and accessible using their unique `@id`. This workflow can be extended for richer analyses as more data is added to the schema.
